In [ ]:
# === Imports ===
import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_blobs
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



#  Utils functions

In [ ]:
def plot_dataset(X, y, axes):
    plt.plot(X[y == 0, 0], X[y == 0, 1], "bs", label="Class 0")
    plt.plot(X[y == 1, 0], X[y == 1, 1], "g^", label="Class 1")
    plt.axis(axes)
    plt.grid(True, which="both")
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$")

def plot_predictions(clf, axes):
    x0s = np.linspace(axes[0], axes[1], 200)
    x1s = np.linspace(axes[2], axes[3], 200)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_grid = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_grid).reshape(x0.shape)

    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)


In [ ]:
def print_confusion_pm1(y_true, y_pred, title="Confusion Matrix (±1 labels)"):
    """
    Confusion matrix printer for labels in {-1, +1}
    -1 is treated as class 0 (negative)
    +1 is treated as class 1 (positive)
    """
    # Convert {-1,+1} → {0,1} for cm calculation only
    y_true_bin = (y_true == 1).astype(int)
    y_pred_bin = (y_pred == 1).astype(int)

    cm = confusion_matrix(y_true_bin, y_pred_bin)
    tn, fp, fn, tp = cm.ravel()

    print(title)
    print("              |  Predicted -1    | Predicted +1")
    print("-------------------------------------------------")
    print(f"Actual -1     |     {tn:3d}         |    {fp:3d}")
    print("-------------------------------------------------")
    print(f"Actual +1     |     {fn:3d}         |    {tp:3d}")
    print("\n")


In [ ]:
def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    # Extract parameters
    w = svm_clf.coef_[0]
    b = svm_clf.intercept_[0]

    # Decision boundary: w0*x0 + w1*x1 + b = 0
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0]/w[1] * x0 - b/w[1]

    # Margins: w0*x0 + w1*x1 + b = ±1
    margin = 1 / w[1]
    gutter_up = decision_boundary + margin
    gutter_down = decision_boundary - margin

    # Plot decision boundary & margins
    plt.plot(x0, decision_boundary, "k-", linewidth=2)
    plt.plot(x0, gutter_up, "k--", linewidth=2)
    plt.plot(x0, gutter_down, "k--", linewidth=2)

    # Plot support vectors
    svs = svm_clf.support_vectors_
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors="#FFAAAA", edgecolors="k")


#  Classic linearly separable SVM dataset


In [ ]:
# Create 2 separable Gaussian blobs
X, y = make_blobs(n_samples=150,
                  centers=[(-2, -2), (2, 2)],
                  cluster_std=0.8,
                  random_state=42)

# Convert labels {0,1} → {-1,+1} to match SVM theory
y_svm = np.where(y == 0, -1, 1)

# Plot the dataset
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_svm, cmap="bwr", edgecolor="k")
plt.title("Classic Linearly-Separable Dataset for SVM")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(alpha=0.3)
plt.show()


#  Hard-margin (large C) linear SVM on the classic dataset

In [ ]:
hard_svm_clf = SVC(kernel="linear", C=1e6)  # very large C ≈ hard margin
hard_svm_clf.fit(X, y_svm)

In [ ]:
plt.figure(figsize=(6, 5))
xmin, xmax = X[:, 0].min() - 1, X[:, 0].max() + 1
plot_svc_decision_boundary(hard_svm_clf, xmin, xmax)

plt.scatter(X[:, 0][y_svm == 1],  X[:, 1][y_svm == 1],  c="g", marker="^", label="Class +1")
plt.scatter(X[:, 0][y_svm == -1], X[:, 1][y_svm == -1], c="b", marker="s", label="Class -1")

plt.title("Hard-Margin SVM (Large C) on Linearly-Separable Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


#  Soft-margin (smaller C) linear SVM on the same dataset


In [ ]:
soft_svm_clf = SVC(kernel="linear", C=0.1)  # smaller C = softer margin
soft_svm_clf.fit(X, y_svm)

In [ ]:
plt.figure(figsize=(6, 5))
xmin, xmax = X[:, 0].min() - 1, X[:, 0].max() + 1

plot_svc_decision_boundary(soft_svm_clf, xmin, xmax)

plt.scatter(X[:, 0][y_svm == 1],  X[:, 1][y_svm == 1],  c="g", marker="^", label="Class +1")
plt.scatter(X[:, 0][y_svm == -1], X[:, 1][y_svm == -1], c="b", marker="s", label="Class -1")

plt.title("Soft-Margin SVM (Smaller C) on the Same Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
y_pred_hard = hard_svm_clf.predict(X)

print_confusion_pm1(y_svm, y_pred_hard, "Hard-Margin SVM")

In [ ]:
y_pred_soft = soft_svm_clf.predict(X)

print_confusion_pm1(y_svm, y_pred_soft, "Soft-Margin SVM")

In [ ]:
# Clean data
X_neg = np.random.randn(75, 2) * 0.6 + np.array([-2, -2])  # purple cluster
X_pos = np.random.randn(75, 2) * 0.6 + np.array([ 2,  2])  # green cluster

X = np.vstack([X_neg, X_pos])
y = np.hstack([-np.ones(75), np.ones(75)])

# THE TWO KILLER OUTLIERS
outlier1 = np.array([[-2.0, -2.0]])   # inside purple cluster → labeled +1 (green)
outlier2 = np.array([[ 2.0,  2.0]])   # inside green cluster  → labeled -1 (purple)

X_dirty = np.vstack([X, outlier1, outlier2])
y_dirty = np.hstack([y, 1, -1])   # ← flipped labels!

# PLOT — exactly what you wanted
plt.figure(figsize=(7, 6))

# Normal points (slightly transparent)
plt.scatter(X[y == -1, 0], X[y == -1, 1], c='#8A2BE2', alpha=0.7, s=60, label='Class -1')
plt.scatter(X[y ==  1, 0], X[y ==  1, 1], c='#32CD32', alpha=0.7, s=60, label='Class +1')

# The two mislabeled outliers — colored by their WRONG label
plt.scatter(outlier1[:,0], outlier1[:,1], c='#32CD32', s=60)
plt.scatter(outlier2[:,0], outlier2[:,1], c='#8A2BE2', s=60)

plt.title("Hard-Margin SVM = Dead", fontsize=16, pad=20)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.xlim(-5, 5)
plt.ylim(-5, 5)
plt.show()

In [ ]:
# hard_svm = SVC(kernel="linear", C=1e12)   # huge C = hard margin
# hard_svm.fit(X_dirty, y_dirty)

In [ ]:
# SOFT-MARGIN SVM THAT ACTUALLY WORKS ON YOUR DIRTY DATA
soft_svm = SVC(kernel="linear", C=1.0)   # C=1.0 is a reasonable soft margin
soft_svm.fit(X_dirty, y_dirty)

print(f"Soft-margin SVM fitted perfectly")
print(f"Number of support vectors: {soft_svm.n_support_.sum()}")  # usually ~6–12
print(f"Misclassified points: {(soft_svm.predict(X_dirty) != y_dirty).sum()}")  # usually 0 or 2

# Plot the TRUTH — soft margin saves the day
plt.figure(figsize=(7, 6))

plot_svc_decision_boundary(soft_svm, xmin=-4.5, xmax=4.5)

# Clean points
plt.scatter(X_dirty[y_dirty == -1, 0], X_dirty[y_dirty == -1, 1],
            c='#8A2BE2', s=60, alpha=0.8, label='Class -1')
plt.scatter(X_dirty[y_dirty ==  1, 0], X_dirty[y_dirty ==  1, 1],
            c='#32CD32', s=60, alpha=0.8, label='Class +1')

# The two killer outliers — now clearly violated but handled gracefully
plt.scatter(outlier1[0,0], outlier1[0,1], c='#32CD32', s=60)
plt.scatter(outlier2[0,0], outlier2[0,1], c='#8A2BE2', s=60)

plt.title("Soft-Margin SVM (C=1.0)\nIgnores the two mislabeled points → wide, sane margin",
          fontsize=14, pad=15)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.3)
plt.axis('equal')
plt.xlim(-4.5, 4.5)
plt.ylim(-4.5, 4.5)
plt.show()

# Kernel SVM

In [ ]:
# Non-linear dataset
X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X_moon[y_moon == 0, 0], X_moon[y_moon == 0, 1], c="b", label="Class 0")
plt.scatter(X_moon[y_moon == 1, 0], X_moon[y_moon == 1, 1], c="g", label="Class 1")
plt.title("Two Moons – Nonlinear Dataset")
plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.legend()
plt.grid(alpha=0.3)
plt.axis("equal")
plt.show()


In [ ]:
linear_svm_moon = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0))
])

linear_svm_moon.fit(X_moon, y_moon)


In [ ]:
plt.figure(figsize=(6, 5))
axes = [-1.5, 2.5, -1, 1.5]
plot_predictions(linear_svm_moon, axes)
plot_dataset(X_moon, y_moon, axes)
plt.title("Linear SVM on Two Moons (fails)")
plt.legend()
plt.show()


In [ ]:
# 1. Get predictions from the linear SVM
y_pred_linear = linear_svm_moon.predict(X_moon)

# 2. Convert {0,1} → {-1,+1} for both true and predicted
y_moon_pm1 = np.where(y_moon == 0, -1, +1)
y_pred_linear_pm1 = np.where(y_pred_linear == 0, -1, +1)

print_confusion_pm1(y_moon_pm1, y_pred_linear_pm1,  title="Linear SVM on Two Moons")


In [ ]:
rbf_svm_moon = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", gamma=5, C=1.0))])
rbf_svm_moon.fit(X_moon, y_moon)


In [ ]:
plt.figure(figsize=(6, 5))
axes = [-1.5, 2.5, -1, 1.5]
plot_predictions(rbf_svm_moon, axes)
plot_dataset(X_moon, y_moon, axes)
plt.title("RBF Kernel SVM on Two Moons")
plt.legend()
plt.show()


In [ ]:
y_pred_rbf = rbf_svm_moon.predict(X_moon)

y_pred_rbf_pm1 = np.where(y_pred_rbf == 0, -1, +1)
y_moon_pm1     = np.where(y_moon == 0, -1, +1)

print_confusion_pm1(
    y_moon_pm1,
    y_pred_rbf_pm1,
    title="RBF Kernel SVM – Confusion Matrix"
)


## And if we want different kernels?

### SVM Kernels Summary and how to write them in sklearn

1. Linear Kernel
$$K(x, z) = xᵀz$$

 Hyperparameters
- C

---

2. Polynomial Kernel
$$K(x, z) = (xᵀz + coef0)^{degree}$$

 Hyperparameters
- degree
- coef0
- C

---

 3. RBF (Gaussian) Kernel
$$K(x, z) = e^{(−γ ||x − z||^2)}$$

 Hyperparameters
- gamma
- C

---

 4. Sigmoid (tanh) Kernel
$$K(x, z) = tanh(γ xᵀz + coef0)$$

 Hyperparameters
- gamma
- coef0
- C


In [ ]:
# Kernel + hyperparameter grids
kernels = {
    "linear": [
        {"C": 0.1},
        {"C": 1},
        {"C": 10}
    ],
    "poly": [
        {"degree": 3, "C": 1, "coef0": 1},
        {"degree": 5, "C": 1, "coef0": 1},
        {"degree": 7, "C": 1, "coef0": 1}
    ],
    "rbf": [
        {"gamma": 0.1, "C": 1},
        {"gamma": 1.0, "C": 1},
        {"gamma": 5.0, "C": 1}
    ],
    "sigmoid": [
        {"gamma": 0.1, "coef0": 0, "C": 1},
        {"gamma": 1.0, "coef0": 1, "C": 1},
        {"gamma": 5.0, "coef0": 1, "C": 1}
    ]
}


In [ ]:
axes_region = [-1.5, 2.5, -1, 1.5]

n_rows = len(kernels)
n_cols = 3  # we chose 3 parameter settings for each kernel
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows),
                         sharex=True, sharey=True)

# Ensure axes is 2D even if n_rows==1
if n_rows == 1:
    axes = np.array([axes])

for row, (kernel_name, param_list) in enumerate(kernels.items()):
    for col, params in enumerate(param_list):

        if kernel_name == "linear":
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(kernel="linear", C=params["C"]))
            ])
            title = f"linear, C={params['C']}"

        elif kernel_name == "poly":
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(
                    kernel="poly",
                    degree=params["degree"],
                    coef0=params["coef0"],
                    C=params["C"],
                )),
            ])
            title = f"poly, d={params['degree']}, C={params['C']}"

        elif kernel_name == "rbf":
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(
                    kernel="rbf",
                    gamma=params["gamma"],
                    C=params["C"],
                )),
            ])
            title = f"rbf, γ={params['gamma']}, C={params['C']}"

        elif kernel_name == "sigmoid":
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("svm", SVC(
                    kernel="sigmoid",
                    gamma=params["gamma"],
                    coef0=params["coef0"],
                    C=params["C"],
                )),
            ])
            title = f"sigmoid, γ={params['gamma']}, coef0={params['coef0']}, C={params['C']}"

        # ---- Fit model ----
        clf.fit(X_moon, y_moon)

        # ---- Plot decision region ----
        ax = axes[row, col]
        plt.sca(ax)
        plot_predictions(clf, axes_region)
        plot_dataset(X_moon, y_moon, axes_region)
        plt.title(title)

        # ---- Compute metrics ----
        y_pred = clf.predict(X_moon)
        acc = accuracy_score(y_moon, y_pred)

        # Convert to ±1 for your confusion printer
        y_true_pm1 = np.where(y_moon == 0, -1, +1)
        y_pred_pm1 = np.where(y_pred == 0, -1, +1)

        print(f"{title} → accuracy = {acc:.3f}")
        print_confusion_pm1(y_true_pm1, y_pred_pm1, title)

plt.tight_layout()
plt.show()


# polynimoal kernel

In [ ]:
# -------------------------
# 1) Data: quadratic decision boundary
# -------------------------
np.random.seed(42)
n = 250

x1 = np.random.uniform(-3, 3, size=n)
# True boundary: x2 = 0.5 * x1^2 - 1
true_boundary = 0.5 * x1**2 - 1

# Generate x2 around the boundary with noise
x2 = true_boundary + np.random.normal(scale=0.8, size=n)

# Label: class 1 above curve, class 0 below
y = (x2 > true_boundary).astype(int)

X = np.c_[x1, x2]

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=25, label="class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=25, label="class 1")
xs = np.linspace(-3, 3, 200)
plt.plot(xs, 0.5 * xs**2 - 1, lw=2, label="true quadratic boundary")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Synthetic data with quadratic decision boundary")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [ ]:
# -------------------------
# 2) Fit poly SVM for various degrees
# -------------------------
def fit_poly_svm_and_scores(degree):
    """Fit poly-kernel SVM of given degree; return model, train_acc, val_acc."""
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="poly", degree=degree, coef0=1.0, C=1.0))
    ])
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_val_pred = clf.predict(X_val)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    return clf, train_acc, val_acc

In [ ]:
max_degree = 60
degrees = np.arange(1, max_degree + 1)
models = {}
train_accs = []
val_accs = []

for d in degrees:
    clf, train_acc, val_acc = fit_poly_svm_and_scores(d)
    models[d] = clf
    train_accs.append(train_acc)
    val_accs.append(val_acc)

train_accs = np.array(train_accs)
val_accs = np.array(val_accs)

best_deg = int(degrees[np.argmax(val_accs)])
under_deg = 1
over_deg = max_degree
print(f"Best degree by validation accuracy: {best_deg}")


In [ ]:
# -------------------------
# 3) Plot underfit vs good vs overfit decision boundaries
# -------------------------
def plot_decision_regions(clf, ax, title, axes_limits=(-3, 3, -4, 6)):
    x0s = np.linspace(axes_limits[0], axes_limits[1], 300)
    x1s = np.linspace(axes_limits[2], axes_limits[3], 300)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_grid = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_grid).reshape(x0.shape)

    ax.contourf(x0, x1, y_pred, alpha=0.25, cmap=plt.cm.brg)
    ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
               s=20, c="tab:blue", label="class 0 (train)")
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
               s=20, c="tab:green", label="class 1 (train)")

    xs = np.linspace(-3, 3, 200)
    ax.plot(xs, 0.5 * xs**2 - 1, "k--", lw=1.5, label="true boundary")
    ax.set_title(title)
    ax.set_xlim(axes_limits[0], axes_limits[1])
    ax.set_ylim(axes_limits[2], axes_limits[3])
    ax.grid(alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cases = [("Underfit", under_deg), ("Good (val-best)", best_deg), ("Overfit", over_deg)]

for ax, (title, d) in zip(axes, cases):
    plot_decision_regions(models[d], ax,
                          f"{title}\npoly degree = {d}")
axes[0].set_ylabel("$x_2$")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# 4) “Elbow” plot: validation accuracy vs degree
# -------------------------
plt.figure(figsize=(15, 6))
plt.plot(degrees, train_accs, "-o", label="train acc")
plt.plot(degrees, val_accs, "-o", label="val acc")
plt.xlabel("Polynomial degree")
plt.ylabel("Accuracy")
plt.title("Model selection for poly-kernel SVM")
plt.grid(alpha=0.3)

# mark best degree
plt.scatter([best_deg], [val_accs[best_deg - 1]], s=80, zorder=3)
plt.annotate(f"best = {best_deg}",
             xy=(best_deg, val_accs[best_deg - 1]),
             xytext=(best_deg + 0.5, val_accs[best_deg - 1] - 0.05),
             arrowprops=dict(arrowstyle="->", lw=1))

plt.legend()
plt.tight_layout()
plt.show()


# RBF kernel

In [ ]:
# -------------------------
# 1) Data: two moons (nice for RBF)
# -------------------------
np.random.seed(42)
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

plt.figure(figsize=(5, 4))
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=25, label="class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=25, label="class 1")
plt.title("Two-moons dataset (nonlinear)")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.grid(alpha=0.3)
plt.axis("equal")
plt.show()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
# -------------------------
# 2) Fit RBF SVM for various gamma
# -------------------------
def fit_rbf_svm_and_scores(gamma, C=1.0):
    """Fit RBF SVM with given gamma (and C); return model, train_acc, val_acc."""
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", gamma=gamma, C=C))
    ])
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_val_pred = clf.predict(X_val)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    return clf, train_acc, val_acc


In [ ]:
# Try gamma on a log-scale range
gammas = np.logspace(-2, 2, 30)  # 0.01 ... 100
C_fixed = 1.0

models = {}
train_accs = []
val_accs = []

for g in gammas:
    clf, train_acc, val_acc = fit_rbf_svm_and_scores(g, C=C_fixed)
    models[g] = clf
    train_accs.append(train_acc)
    val_accs.append(val_acc)

train_accs = np.array(train_accs)
val_accs = np.array(val_accs)

best_gamma = float(gammas[np.argmax(val_accs)])
gamma_under = float(gammas[0])      # very small -> underfit
gamma_over  = float(gammas[-1])     # very large -> overfit

print(f"Best gamma by validation accuracy: {best_gamma:.4f}")

In [ ]:
# -------------------------
# 3) Plot underfit vs good vs overfit decision boundaries
# -------------------------
def plot_decision_regions(clf, ax, title, axes_limits=(-2, 3, -1.5, 2.0)):
    x0s = np.linspace(axes_limits[0], axes_limits[1], 300)
    x1s = np.linspace(axes_limits[2], axes_limits[3], 300)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_grid = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_grid).reshape(x0.shape)

    ax.contourf(x0, x1, y_pred, alpha=0.25, cmap=plt.cm.brg)
    ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
               s=20, c="tab:blue", label="class 0 (train)")
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
               s=20, c="tab:green", label="class 1 (train)")
    ax.set_title(title)
    ax.set_xlim(axes_limits[0], axes_limits[1])
    ax.set_ylim(axes_limits[2], axes_limits[3])
    ax.grid(alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cases = [
    ("Underfit", gamma_under),
    ("Good (val-best)", best_gamma),
    ("Overfit", gamma_over),
]

for ax, (title, g) in zip(axes, cases):
    clf = models[g]
    plot_decision_regions(
        clf,
        ax,
        f"{title}\nRBF gamma = {g:.4g}, C = {C_fixed}"
    )

axes[0].set_ylabel("$x_2$")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------
# 4) “Elbow” plot: accuracy vs log10(gamma)
# -------------------------
plt.figure(figsize=(12, 6))
log_g = np.log10(gammas)
plt.plot(log_g, train_accs, "-o", label="train acc")
plt.plot(log_g, val_accs, "-o", label="val acc")
plt.xlabel("log10(gamma)")
plt.ylabel("Accuracy")
plt.title("Model selection for RBF-kernel SVM")
plt.grid(alpha=0.3)

# mark best gamma
best_idx = np.argmax(val_accs)
plt.scatter([log_g[best_idx]], [val_accs[best_idx]], s=80, zorder=3)
plt.annotate(
    f"best γ ≈ {gammas[best_idx]:.3g}",
    xy=(log_g[best_idx], val_accs[best_idx]),
    xytext=(log_g[best_idx] + 0.3, val_accs[best_idx] - 0.05),
    arrowprops=dict(arrowstyle="->", lw=1),
)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# 1) Data: S-shaped tanh decision boundary
# -------------------------
np.random.seed(42)
n = 300

x1 = np.random.uniform(-4, 4, size=n)

# true boundary: tanh-shaped curve in x2
true_boundary = 2.0 * np.tanh(x1)      # between about -2 and +2
x2 = true_boundary + np.random.normal(scale=0.6, size=n)

# class 1 above the curve, class 0 below
y = (x2 > true_boundary).astype(int)
X = np.c_[x1, x2]

plt.figure(figsize=(5, 4))
plt.scatter(X[y == 0, 0], X[y == 0, 1], s=25, label="class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], s=25, label="class 1")
xs = np.linspace(-4, 4, 400)
plt.plot(xs, 2.0 * np.tanh(xs), "k--", lw=2, label="true tanh boundary")
plt.title("Synthetic data with tanh-shaped decision boundary")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# -------------------------
# 2) Fit sigmoid-kernel SVM for various gamma
# -------------------------
def fit_sigmoid_svm_and_scores(gamma, coef0=1.0, C=1.0):
    """
    Fit sigmoid (tanh) kernel SVM with given gamma, coef0, C;
    return model, train_acc, val_acc.
    """
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="sigmoid", gamma=gamma, coef0=coef0, C=C))
    ])
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_val_pred = clf.predict(X_val)
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    return clf, train_acc, val_acc

gammas = np.logspace(-2, 1, 10)   # 0.01 ... 10
C_fixed = 1.0
coef0_fixed = 1.0

models = {}
train_accs = []
val_accs = []

for g in gammas:
    clf, train_acc, val_acc = fit_sigmoid_svm_and_scores(
        gamma=g, coef0=coef0_fixed, C=C_fixed
    )
    models[g] = clf
    train_accs.append(train_acc)
    val_accs.append(val_acc)

train_accs = np.array(train_accs)
val_accs = np.array(val_accs)

best_gamma = float(gammas[np.argmax(val_accs)])
gamma_under = float(gammas[0])   # too small => almost linear (underfit)
gamma_over  = float(gammas[-1])  # too large => very wiggly, overfit-ish

print(f"Best gamma by validation accuracy (sigmoid kernel): {best_gamma:.4f}")

# -------------------------
# 3) Plot underfit vs good vs overfit decision boundaries
# -------------------------
def plot_decision_regions(clf, ax, title, axes_limits=(-4, 4, -4, 4)):
    x0s = np.linspace(axes_limits[0], axes_limits[1], 300)
    x1s = np.linspace(axes_limits[2], axes_limits[3], 300)
    x0, x1 = np.meshgrid(x0s, x1s)
    X_grid = np.c_[x0.ravel(), x1.ravel()]
    y_pred = clf.predict(X_grid).reshape(x0.shape)

    ax.contourf(x0, x1, y_pred, alpha=0.25, cmap=plt.cm.brg)
    ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
               s=20, c="tab:blue", label="class 0 (train)")
    ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
               s=20, c="tab:green", label="class 1 (train)")

    xs = np.linspace(-4, 4, 400)
    ax.plot(xs, 2.0 * np.tanh(xs), "k--", lw=1.5, label="true tanh boundary")

    ax.set_title(title)
    ax.set_xlim(axes_limits[0], axes_limits[1])
    ax.set_ylim(axes_limits[2], axes_limits[3])
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
cases = [
    ("Underfit", gamma_under),
    ("Good (val-best)", best_gamma),
    ("Overfit", gamma_over),
]

for ax, (title, g) in zip(axes, cases):
    clf = models[g]
    plot_decision_regions(
        clf,
        ax,
        f"{title}\nsigmoid γ = {g:.3g}, coef0 = {coef0_fixed}, C = {C_fixed}"
    )

axes[0].set_ylabel("$x_2$")
axes[1].legend(loc="upper left")
plt.tight_layout()
plt.show()

# -------------------------
# 4) “Elbow” / selection plot: accuracy vs log10(gamma)
# -------------------------
plt.figure(figsize=(6, 4))
log_g = np.log10(gammas)
plt.plot(log_g, train_accs, "-o", label="train acc")
plt.plot(log_g, val_accs, "-o", label="val acc")
plt.xlabel("log10(gamma)")
plt.ylabel("Accuracy")
plt.title("Model selection for sigmoid-kernel SVM")
plt.grid(alpha=0.3)

best_idx = np.argmax(val_accs)
plt.scatter([log_g[best_idx]], [val_accs[best_idx]], s=80, zorder=3)
plt.annotate(
    f"best γ ≈ {gammas[best_idx]:.3g}",
    xy=(log_g[best_idx], val_accs[best_idx]),
    xytext=(log_g[best_idx] + 0.3, val_accs[best_idx] - 0.05),
    arrowprops=dict(arrowstyle="->", lw=1),
)

plt.legend()
plt.tight_layout()
plt.show()


# Custom kernel

In [ ]:
plt.figure(figsize=(6, 5))
axes = [-1.5, 2.5, -1, 1.5]
plot_dataset(X_moon, y_moon, axes)
plt.title("Custom kernel")
plt.legend()
plt.show()


In [ ]:
gamma_custom = 0.5  # for the RBF part

def linear_plus_rbf_kernel(X, Y):
    """
    Custom kernel: K(x,z) = x^T z + exp(-gamma ||x - z||^2)
    X: (n_samples_1, d)
    Y: (n_samples_2, d)
    returns: (n_samples_1, n_samples_2)
    """
    # linear part
    linear = X @ Y.T

    # squared Euclidean distances for RBF part
    X_norm2 = np.sum(X**2, axis=1)[:, None]       # (n1, 1)
    Y_norm2 = np.sum(Y**2, axis=1)[None, :]       # (1, n2)
    sq_dists = X_norm2 + Y_norm2 - 2 * linear     # (n1, n2)

    rbf = np.exp(-gamma_custom * sq_dists)

    return linear + rbf


In [ ]:
custom_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel=linear_plus_rbf_kernel, C=1.0))
])

custom_svm.fit(X_moon, y_moon)


In [ ]:
plt.figure(figsize=(6, 5))
axes = [-1.5, 2.5, -1, 1.5]
plot_predictions(custom_svm, axes)
plot_dataset(X_moon, y_moon, axes)
plt.title("Custom kernel: linear + RBF")
plt.legend()
plt.show()


## Inline Question:

Now that you have trained a good model, you may find that your testing accuracy is much lower than the training accuracy. In what ways can we decrease this gap? Select all that apply.

1. Train on a larger dataset.
2. Add more hidden units.
3. Increase the regularization strength.
4. None of the above.
